In [156]:
import numpy as np
import cv2
import pandas as pd
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [157]:
df_train = pd.read_csv('train.csv')
df_test= pd.read_csv('test.csv')

In [158]:
df_train

,sample_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,target_name,target
0,ID1011485656__Dry_Clover_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Clover_g,0.0000
1,ID1011485656__Dry_Dead_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Dead_g,31.9984
2,ID1011485656__Dry_Green_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Green_g,16.2751
3,ID1011485656__Dry_Total_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Total_g,48.2735
4,ID1011485656__GDM_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,GDM_g,16.2750
...,...,...,...,...,...,...,...,...,...
1780,ID983582017__Dry_Clover_g,train/ID983582017.jpg,2015/9/1,WA,Ryegrass,0.64,9.0000,Dry_Clover_g,0.0000
1781,ID983582017__Dry_Dead_g,train/ID983582017.jpg,2015/9/1,WA,Ryegrass,0.64,9.0000,Dry_Dead_g,0.0000
1782,ID983582017__Dry_Green_g,train/ID983582017.jpg,2015/9/1,WA,Ryegrass,0.64,9.0000,Dry_Green_g,40.9400
1783,ID983582017__Dry_Total_g,train/ID983582017.jpg,2015/9/1,WA,Ryegrass,0.64,9.0000,Dry_Total_g,40.9400


In [159]:
df_test

,sample_id,image_path,target_name
0,ID1001187975__Dry_Clover_g,test/ID1001187975.jpg,Dry_Clover_g
1,ID1001187975__Dry_Dead_g,test/ID1001187975.jpg,Dry_Dead_g
2,ID1001187975__Dry_Green_g,test/ID1001187975.jpg,Dry_Green_g
3,ID1001187975__Dry_Total_g,test/ID1001187975.jpg,Dry_Total_g
4,ID1001187975__GDM_g,test/ID1001187975.jpg,GDM_g


In [160]:
os.listdir('train/')

['ID1011485656.jpg',
 'ID1012260530.jpg',
 'ID1025234388.jpg',
 'ID1028611175.jpg',
 'ID1035947949.jpg',
 'ID1036339023.jpg',
 'ID1049634115.jpg',
 'ID1051144034.jpg',
 'ID1052620238.jpg',
 'ID105271783.jpg',
 'ID1053972079.jpg',
 'ID1058383417.jpg',
 'ID1062837331.jpg',
 'ID1070112260.jpg',
 'ID1078930021.jpg',
 'ID1084819986.jpg',
 'ID1088965591.jpg',
 'ID1098771283.jpg',
 'ID1103883611.jpg',
 'ID1108283583.jpg',
 'ID1113121340.jpg',
 'ID1113329413.jpg',
 'ID1119739385.jpg',
 'ID1119761112.jpg',
 'ID1121692672.jpg',
 'ID1127246618.jpg',
 'ID112966473.jpg',
 'ID1131079710.jpg',
 'ID1136169672.jpg',
 'ID1139866256.jpg',
 'ID1139918758.jpg',
 'ID1140993511.jpg',
 'ID1148528732.jpg',
 'ID1148666289.jpg',
 'ID1149598723.jpg',
 'ID1159071020.jpg',
 'ID1163061745.jpg',
 'ID1168534540.jpg',
 'ID1176292407.jpg',
 'ID1182523622.jpg',
 'ID1183807388.jpg',
 'ID1193692654.jpg',
 'ID1199150612.jpg',
 'ID1208644039.jpg',
 'ID1211362607.jpg',
 'ID121331988.jpg',
 'ID1215977190.jpg',
 'ID1217108125.j

In [161]:
df_train = df_train[["image_path", "target_name", "target"]]

In [162]:
df_train.head()

,image_path,target_name,target
0,train/ID1011485656.jpg,Dry_Clover_g,0.0000
1,train/ID1011485656.jpg,Dry_Dead_g,31.9984
2,train/ID1011485656.jpg,Dry_Green_g,16.2751
3,train/ID1011485656.jpg,Dry_Total_g,48.2735
4,train/ID1011485656.jpg,GDM_g,16.2750


In [163]:
X_csv = df_train[["image_path", "target_name"]]

In [164]:
y_csv = df_train['target']

In [165]:
X_csv

,image_path,target_name
0,train/ID1011485656.jpg,Dry_Clover_g
1,train/ID1011485656.jpg,Dry_Dead_g
2,train/ID1011485656.jpg,Dry_Green_g
3,train/ID1011485656.jpg,Dry_Total_g
4,train/ID1011485656.jpg,GDM_g
...,...,...
1780,train/ID983582017.jpg,Dry_Clover_g
1781,train/ID983582017.jpg,Dry_Dead_g
1782,train/ID983582017.jpg,Dry_Green_g
1783,train/ID983582017.jpg,Dry_Total_g


In [166]:
y_csv

0        0.0000
1       31.9984
2       16.2751
3       48.2735
4       16.2750
         ...   
1780     0.0000
1781     0.0000
1782    40.9400
1783    40.9400
1784    40.9400
Name: target, Length: 1785, dtype: float64

In [167]:
from sklearn.preprocessing import LabelEncoder

In [168]:
label = LabelEncoder()

In [169]:
X_csv['target_name'] = label.fit_transform(X_csv['target_name'])

C:\Users\USER\AppData\Local\Temp\ipykernel_12904\2820114327.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_csv['target_name'] = label.fit_transform(X_csv['target_name'])


In [170]:
X_csv

,image_path,target_name
0,train/ID1011485656.jpg,0
1,train/ID1011485656.jpg,1
2,train/ID1011485656.jpg,2
3,train/ID1011485656.jpg,3
4,train/ID1011485656.jpg,4
...,...,...
1780,train/ID983582017.jpg,0
1781,train/ID983582017.jpg,1
1782,train/ID983582017.jpg,2
1783,train/ID983582017.jpg,3


In [171]:
label = {
'Dry_Clover_g': 0,
'Dry_Dead_g': 1,
'Dry_Green_g': 2,
'Dry_Total_g': 3,
'GDM_g': 4
}

In [172]:
df = pd.read_csv("train.csv")

X = []
y = []

for i in range(len(df)):
    
    img_path = df["image_path"][i]
    label = df["target"][i]

    image = cv2.imread(img_path)
    image = cv2.resize(image,(224,224))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    image = preprocess_input(image)
    X.append(image)
    y.append(label)

X = np.array(X)
y = np.array(y)

In [173]:
X.shape

(1785, 224, 224, 3)

In [174]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [175]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

In [176]:
train_generator = datagen.flow(
    X_train,
    y_train,
    batch_size=32
)

In [177]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)
for layer in base_model.layers:
    layer.trainable = False

In [185]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

In [186]:
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)

predictions = Dense(1)(x)

model = Model(inputs=base_model.input, outputs=predictions)

In [187]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
    loss='mse',
    metrics=['mae']
)


In [188]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [189]:
history = model.fit(
    train_generator,
    validation_data=(X_val, y_val),
    epochs=10,
    callbacks=[early_stop]
)

Epoch 1/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 25s 406ms/step - loss: 1270.1179 - mae: 24.5946 - val_loss: 1329.3549 - val_mae: 25.6118
Epoch 2/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 17s 380ms/step - loss: 1252.0231 - mae: 24.3242 - val_loss: 1320.3003 - val_mae: 25.4789
Epoch 3/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 17s 378ms/step - loss: 1233.4216 - mae: 24.0341 - val_loss: 1310.0011 - val_mae: 25.3326
Epoch 4/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 17s 385ms/step - loss: 1213.7134 - mae: 23.7628 - val_loss: 1298.9523 - val_mae: 25.1768
Epoch 5/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 17s 384ms/step - loss: 1192.0981 - mae: 23.4675 - val_loss: 1286.7013 - val_mae: 25.0035
Epoch 6/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 18s 398ms/step - loss: 1170.0891 - mae: 23.1578 - val_loss: 1273.7629 - val_mae: 24.8202
Epoch 7/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 18s 400ms/step - loss: 1147.9832 - mae: 22.8286 - val_loss: 1259.6520 - val_mae: 24.6163
Epoch 8/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 18s 408ms/step - loss: 1121.0752 - mae: 22.4705 - val_loss: 1243.6230 - v

In [190]:
for layer in base_model.layers[-15:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="mse",
    metrics=["mae"]
)

history = model.fit(
    train_generator,
    validation_data=(X_val, y_val),
    epochs=10,
    callbacks=[early_stop]
)

Epoch 1/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 25s 408ms/step - loss: 1027.4534 - mae: 21.3254 - val_loss: 1180.9449 - val_mae: 23.5029
Epoch 2/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 17s 381ms/step - loss: 977.1570 - mae: 20.7428 - val_loss: 1150.9163 - val_mae: 23.0955
Epoch 3/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 17s 385ms/step - loss: 932.4532 - mae: 20.2652 - val_loss: 1118.4503 - val_mae: 22.6844
Epoch 4/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 18s 403ms/step - loss: 879.0186 - mae: 19.7423 - val_loss: 1085.6368 - val_mae: 22.2983
Epoch 5/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 18s 401ms/step - loss: 853.5809 - mae: 19.3970 - val_loss: 1052.2806 - val_mae: 21.9149
Epoch 6/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 18s 402ms/step - loss: 804.3895 - mae: 19.0193 - val_loss: 1019.4728 - val_mae: 21.5656
Epoch 7/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 18s 404ms/step - loss: 761.4905 - mae: 18.7080 - val_loss: 989.9865 - val_mae: 21.2681
Epoch 8/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 18s 405ms/step - loss: 742.9274 - mae: 18.4847 - val_loss: 961.0312 - val_mae: 2